In [1]:
import tomli
%env JAX_PLATFORMS=cpu
%load_ext autoreload

import numpy as onp

import matplotlib.pyplot as plt
import matplotlib as mpl

import sys
sys.path.append('../scripts')

mpl.rcParams['text.usetex'] = True

from pathlib import Path######

In [2]:
%autoreload 2

import eval_utils
import visualize

In [14]:
train_dir = Path("../models/titanium_train_solid_MACE_r_cutoff_0.5_2025_11_13_f15304d6-4928-486b-b4e7-1fa652dadbff")

In [15]:
trainer = onp.load(train_dir / "trainer.pkl", allow_pickle=True)


In [25]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3), layout="constrained")

epochs = len(trainer["epoch_losses"])

ax1.semilogy(onp.linspace(0, epochs, len(trainer["batch_losses"])), trainer["batch_losses"])
ax1.semilogy(trainer["epoch_losses"])
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.set_yscale("linear")
ax1.set_ylim([0, 3.5])

ax2.plot(onp.linspace(0, epochs, len(trainer["batch_gradient_norms"])), trainer["batch_gradient_norms"])
ax2.semilogy(trainer["gradient_norm_history"])
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Gradient Norm")
ax2.set_yscale("linear")
ax2.yaxis.set_major_formatter("{x:.0e}")

fig.legend(["Epoch", "Batch"], loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2)

fig.savefig(Path("../plots") / "loss.pdf", bbox_inches="tight")


In [18]:
fig, axes = plt.subplots(4, 3, figsize=(7, 5), layout="constrained")
axes = axes.ravel()
for idx in range(12):
    keys = trainer["predictions"][idx].keys()
    axes[idx].plot(trainer["predictions"][idx][max(keys)]["rdf"])
    axes[idx].plot(trainer["predictions"][idx + 12][max(keys)]["rdf"])

In [19]:
fig, axes = plt.subplots(4, 3, figsize=(7, 5), layout="constrained")
axes = axes.ravel()
for idx in range(12):
    keys = trainer["predictions"][idx].keys()
    axes[idx].plot(trainer["predictions"][idx][min(keys)]["rdf"])
    axes[idx].plot(trainer["predictions"][idx][max(keys)]["rdf"])

In [20]:
fig, axes = plt.subplots(4, 3, figsize=(7, 5), layout="constrained")
axes = axes.ravel()

rdf = onp.linspace(0, 0.9, 150)
for idx in range(12):
    fmt = "Epoch {}" if idx == 0 else "_{}"
    keys = trainer["predictions"][idx].keys()
    axes[idx].plot(rdf, trainer["predictions"][idx + 12][min(keys)]["rdf"], label=fmt.format(min(keys)))
    axes[idx].plot(rdf, trainer["predictions"][idx + 12][max(keys)]["rdf"], label=fmt.format(max(keys)))

    # if idx % 3 == 0:
    #     axes[idx].set_ylabel("RDF")
    # if idx >= 3:
    #     axes[idx].set_xlabel("Distance [nm]")

fig.legend(loc="outside upper center", ncols=2)

In [21]:
fig, axes = plt.subplots(4, 3, figsize=(7, 8), layout="constrained", sharex=True, sharey=True)
axes = axes.ravel()

with open(train_dir / "config.toml", "rb") as f:
  config = tomli.load(f)

errs = 0.0
for idx in range(12):
    keys = trainer["predictions"][idx].keys()

    pred_solid = onp.asarray([trainer["predictions"][idx][k]["free_energy"] for k in range(max(keys))])
    pred_liquid = onp.asarray([trainer["predictions"][idx + 12][k]["free_energy"] for k in range(max(keys))])

    mean = (pred_solid + pred_liquid) / 2
    diff = (pred_solid - pred_liquid)
    err = onp.abs(diff[-1] - config["targets"]["free_energy_diff"][idx])

    axes[idx].plot(pred_solid - 0.0 * mean)
    axes[idx].plot(pred_liquid - 0.0 * mean)
    axes[idx].plot(mean - config["targets"]["free_energy_diff"][idx] / 2, "--k", label=f'Target: {config["targets"]["free_energy_diff"][idx]:.2f} kJ/mol\nLearned: {diff[-1]:.2f} kJ/mol\n Error: {err:.2f} kJ/mol')
    axes[idx].plot(mean, ":k")
    axes[idx].plot(mean + config["targets"]["free_energy_diff"][idx] / 2, "--k")
    axes[idx].legend(loc="best", fontsize=8)
    axes[idx].set_ylim([-1600, 400])

    errs += err

    # if idx % 3 == 0:
    #     axes[idx].set_ylabel("Free Energy [eV]")
    # if idx >= 3:
    #     axes[idx].set_xlabel("Epoch")

fig.legend(["Free Energy I", "Free Energy II"], loc="lower center", fontsize=8, bbox_to_anchor=(0.5, 1.0), ncol=2)
fig.supxlabel("Epoch")
fig.supylabel("Free Energy [kJ/mol]")
fig.suptitle(f"MAE: {errs/12:.2f} kJ/mol")


In [23]:
fig, axes = plt.subplots(4, 3, figsize=(7, 5), layout="constrained", sharex=True, sharey=True)
axes = axes.ravel()
for idx in range(12):
    keys = trainer["predictions"][idx].keys()
    axes[idx].plot([trainer["predictions"][idx][k]["pressure"] * 0.001661 for k in range(max(keys))])
    axes[idx].plot([trainer["predictions"][idx + 12][k]["pressure"] * 0.001661 for k in range(max(keys))])
    axes[idx].axhline(idx % 6, color="k", linestyle="--")

    if idx % 3 == 0:
        axes[idx].set_ylabel("Pressure [GPa]")
    if idx >= 3:
        axes[idx].set_xlabel("Epoch")


In [24]:
fig, axes = plt.subplots(4, 3, figsize=(7, 5), layout="constrained", sharex=True, sharey=True)
axes = axes.ravel()
errs = 0.0
for idx in range(12):
    keys = trainer["predictions"][idx].keys()
    axes[idx].plot([trainer["predictions"][idx][k]["pressure"] * 0.001661 - idx % 6 for k in range(max(keys))])
    axes[idx].plot([trainer["predictions"][idx + 12][k]["pressure"] * 0.001661 - idx % 6 for k in range(max(keys))])
    axes[idx].axhline(0, color="k", linestyle="--")

    errs += onp.abs(trainer["predictions"][idx][max(keys)]["pressure"] * 0.001661 - idx % 6)
    errs += onp.abs(trainer["predictions"][idx + 12][max(keys)]["pressure"] * 0.001661 - idx % 6)

    if idx % 3 == 0:
        axes[idx].set_ylabel("Pressure [GPa]")
    if idx >= 3:
        axes[idx].set_xlabel("Epoch")

fig.legend([r"Pressure Difference $\mathrm{\MakeUppercase{\romannumeral 1}}$", r"Pressure Difference $\mathrm{\MakeUppercase{\romannumeral 2}}$"], loc="lower center", ncol=2, bbox_to_anchor=(0.5, 1.0))
plt.suptitle(f"MAE: {errs/24:.3f} GPa")